# FloodTwin Patna — 00 · Setup & build the city model

Runs the heavy, run-rarely stage: download satellite layers, detect sinks, rank recharge
sites, score against radar, and train the differentiable twin. Persists an **artifact
bundle** to Drive that the daily alert + chat agent then consume.

**Runtime:** set **Runtime → Change runtime type → GPU** (needed for the twin in the last cell).

**Before you run:** register for Earth Engine (earthengine.google.com), create a Google Cloud
project, and put its id in `VARUNA_PROJECT_ID` below.

In [ ]:
# 1. Get the package onto Colab. Either clone your repo or upload the `varuna/` folder.
# !git clone https://github.com/<you>/project-varuna.git /content/varuna_repo
import sys; sys.path.insert(0, '/content/varuna_repo')   # <-- folder that CONTAINS `varuna/`

# 2. Install dependencies
!pip install -q pysheds rasterio geopandas shapely osmnx folium earthengine-api pyyaml

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')                      # outputs persist to Drive

os.environ['VARUNA_PROJECT_ID'] = 'your-cloud-project-id'   # <-- EDIT: your Earth Engine project
# os.environ['VARUNA_WORK'] = '/content/drive/MyDrive/floodtwin'   # default when Drive is mounted

import logging; logging.basicConfig(level=logging.INFO, format='%(name)s: %(message)s')
from varuna.config import CFG
from varuna.ee_auth import init_ee
print('work dir:', CFG.work)
init_ee()                                          # interactive OAuth on first run

## 1 · Sinks (nb01) — where does water pool?

In [ ]:
from varuna.build import sinks
top = sinks.run()
top.head(10)

## 2 · Recharge sites (nb02)
⚠️ Replace `gw_levels.csv` in the work dir with **real** pre-monsoon CGWB data from
India-WRIS before trusting the ranking — a SAMPLE file is written automatically otherwise.

In [ ]:
from varuna.build import recharge
ranked = recharge.run()
ranked.head(10)[['sink_id','lat','lon','volume_m3','gw_depth_m','ksat_mm_hr','rsi','recharge_score']]

## 3 · SAR validation (nb03)
First list available Sentinel-1 passes, then pick a date right after heavy rain and re-run.

In [ ]:
from varuna.build import validate
validate.run(event_date=None)            # prints available passes

In [ ]:
scores = validate.run(event_date='2025-07-15')   # <-- EDIT: a date from the list above
print('CSI (credibility metric):', round(scores['csi'], 3), scores)

## 4 · Train the differentiable twin (nb05) — needs GPU

In [ ]:
import torch; assert torch.cuda.is_available(), 'Switch Runtime to GPU first.'
from varuna.build import twin
meta = twin.train_twin()      # ~10-20 min on a T4; saves emulator.pt + twin_meta.pt
print('emulator val RMSE:', round(meta['val_rmse_m']*100,1), 'cm')

In [ ]:
from varuna.io import bundle_status
bundle_status(CFG.work)        # confirm every artifact is present